In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np # Added numpy import

sns.set_theme(style="whitegrid")

# --- Step 1: Ensure DataFrames are loaded (assuming they are already in your environment) ---
# If not, you would load them here, e.g.:
TSMC_historical = pd.read_excel('/content/TSMC_historical.xlsx')
TSM_adr_historical = pd.read_excel('/content/TSM_adr_historical.xlsx')
TWD_USD_historical = pd.read_excel('/content/TWD_USD_historical.xlsx')

# --- Step 2: Standardize and prepare data ---

# Function to clean and standardize a dataframe
def prepare_df(df, date_col='Date', value_col=None, name=None):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    if value_col and name:
        df = df[[value_col]].rename(columns={value_col: name})
    return df

# Prepare each dataframe (adjust 'Date' and 'Close'/'Rate' column names if needed)
# For TSMC Taiwan stock (e.g., assuming 'Last Price' column)
TSMC_historical_processed = prepare_df(TSMC_historical, date_col='Date', value_col='Last Price', name='TSMC_TW_Close')

# For TSM US ADR stock (e.g., assuming 'Last Price' column)
TSM_adr_historical_processed = prepare_df(TSM_adr_historical, date_col='Date', value_col='Last Price', name='TSM_ADR_Close_USD')

# For TWD/USD exchange rate (e.g., assuming 'Last Price' column)
# Assuming the 'Last Price' in TWD_USD_historical is actually USD per X TWD
TWD_USD_historical_processed = prepare_df(TWD_USD_historical, date_col='Date', value_col='Last Price', name='USD_per_TWD_Rate')

# --- Step 3: Merge all data into a single DataFrame ---

# Merge sequentially, starting with TSMC Taiwan
merged_data = pd.merge(TSMC_historical_processed, TSM_adr_historical_processed,
                       left_index=True, right_index=True, how='outer')

# Merge with TWD/USD exchange rate
merged_data = pd.merge(merged_data, TWD_USD_historical_processed,
                       left_index=True, right_index=True, how='outer')

# Forward fill missing values then drop any remaining NaNs (e.g., if a date range is entirely missing)
merged_data = merged_data.ffill().dropna()

# --- Step 4: Calculate TSM ADR equivalent in TWD ---
# TSM ADR price in TWD per underlying share = (TSM_ADR_Close_USD / USD_per_TWD_Rate) / X
# (Since X ADR = X underlying shares)
merged_data['TSM_ADR_Close_TWD'] = (merged_data['TSM_ADR_Close_USD'] / merged_data['USD_per_TWD_Rate']) / 5

# --- Step 5: Plot the processed data ---

fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(14, 15), sharex=True)

# Plot TSMC Taiwan Close Price
axes[0].plot(merged_data.index, merged_data['TSMC_TW_Close'], label='TSMC (Taiwan) Close Price (TWD)', color='blue')
axes[0].set_ylabel('Price (TWD)')
axes[0].set_title('TSMC Taiwan Stock Price Historical Data')
axes[0].legend()

# Plot TSM ADR Close Price (USD)
axes[1].plot(merged_data.index, merged_data['TSM_ADR_Close_USD'], label='TSM ADR (US) Close Price (USD)', color='green')
# axes[X].plot(merged_data.index, merged_data['TSM_ADR_Close_TWD'], label='TSM ADR (US) Close Price (TWD)', color='purple', linestyle='--') # Removed as requested
axes[1].set_ylabel('Price')
axes[1].set_title('TSM US ADR Historical Data (USD)')
axes[1].legend()

# Plot TWD/USD Exchange Rate
axes[2].plot(merged_data.index, merged_data.rename(columns={'USD_per_TWD_Rate': 'TWD_per_USD_Rate'})['TWD_per_USD_Rate'], label='USD/TWD Exchange Rate', color='red') # Corrected label
axes[2].set_ylabel('Rate (USD/TWD)') # Corrected label
axes[2].set_title('USD/TWD Exchange Rate Historical Data') # Corrected label
axes[2].legend()

plt.xlabel('Date')
plt.tight_layout()
plt.show()

print("Data processing and plotting complete. The combined data is available in the 'merged_data' DataFrame.")
print(merged_data.head())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(merged_data.index, merged_data['TSMC_TW_Close'], label='TSMC (Taiwan) Close Price (TWD)', color='blue')
ax.plot(merged_data.index, merged_data['TSM_ADR_Close_TWD'], label='TSM ADR (US) Close Price (TWD)', color='purple', linestyle='--')

ax.set_title('Comparison of TSMC (Taiwan) vs. TSM ADR (US) in TWD')
ax.set_xlabel('Date')
ax.set_ylabel('Price (TWD)')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate the dollar difference
merged_data['Difference_TWD'] = merged_data['TSM_ADR_Close_TWD'] - merged_data['TSMC_TW_Close']

# Calculate the percentage difference
# (ADR_TWD - TW_Close) / TW_Close * X
merged_data['Percentage_Difference'] = ((merged_data['TSM_ADR_Close_TWD'] - merged_data['TSMC_TW_Close']) / merged_data['TSMC_TW_Close']) * 100

fig, axes = plt.subplots(nrows=2, ncols=1, figsize=(14, 10), sharex=True)

# Plot the dollar difference
axes[0].plot(merged_data.index, merged_data['Difference_TWD'], label='ADR - 2330 (TWD)', color='orange')
axes[0].axhline(0, color='grey', linestyle='--', linewidth=0.8) # Add a zero line
axes[0].set_ylabel('Difference (TWD)')
axes[0].set_title('Difference between TSM ADR (TWD/share) and TSMC Taiwan (2330) - Dollar Amount')
axes[0].legend()
axes[0].grid(True)

# Plot the percentage difference
axes[1].plot(merged_data.index, merged_data['Percentage_Difference'], label='ADR - 2330 (%)', color='red')
axes[1].axhline(0, color='grey', linestyle='--', linewidth=0.8) # Add a zero line
axes[1].set_ylabel('Difference (%)')
axes[1].set_title('Difference between TSM ADR (TWD/share) and TSMC Taiwan (2330) - Percentage')
axes[1].legend()
axes[1].grid(True)

plt.xlabel('Date')
plt.tight_layout()
plt.show()

print("Dollar and percentage differences calculated and plotted.")
print(merged_data[['TSMC_TW_Close', 'TSM_ADR_Close_TWD', 'Difference_TWD', 'Percentage_Difference']].head())

### Rolling Z-Score Analysis

To understand the relative performance and volatility of the TSMC Taiwan stock and the TSM ADR, we will calculate and plot their rolling Z-scores. The Z-score measures how many standard deviations an observation is from the mean. A rolling Z-score helps to normalize the series and make them comparable, especially when they have different scales or trends.

We will use a rolling window of X days for this calculation.

In [ ]:
# Define the rolling window size
window_size = 30 # For example, X days

# Calculate the difference in TWD (ADR - TW stock), which we already have as 'Difference_TWD'
# If 'Difference_TWD' was not calculated, it would be: merged_data['TSM_ADR_Close_TWD'] - merged_data['TSMC_TW_Close']

# Calculate rolling mean and standard deviation for the Difference_TWD (ADR Premium)
merged_data['ADR_Premium_RollingMean'] = merged_data['Difference_TWD'].rolling(window=window_size).mean()
merged_data['ADR_Premium_RollingStd'] = merged_data['Difference_TWD'].rolling(window=window_size).std()

# Calculate rolling Z-score for the ADR Premium
merged_data['ADR_Premium_ZScore'] = (merged_data['Difference_TWD'] - merged_data['ADR_Premium_RollingMean']) / merged_data['ADR_Premium_RollingStd']

print(f"Rolling Z-score for ADR Premium calculated with a window size of {window_size} days.")
print(merged_data[['Difference_TWD', 'ADR_Premium_ZScore']].tail())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(merged_data.index, merged_data['ADR_Premium_ZScore'], label='ADR Premium Z-Score (TWD)', color='purple')

ax.axhline(0, color='grey', linestyle='-', linewidth=0.8) # Add a zero line
ax.set_title(f'Rolling {window_size}-Day Z-Score of ADR Premium: TSM ADR vs. TSMC Taiwan (TWD)')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling Z-Score')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Realized Volatility Analysis for TWD/USD Exchange Rate

Realized volatility is a measure of the variability of an asset's price over a specific period, calculated from historical price data. We will calculate the realized volatility for the TWD/USD exchange rate using daily log returns and a rolling window of X days (approximately one month of trading days). The daily volatility will then be annualized.

In [ ]:
# First, resample merged_data to a daily frequency, taking the last observation of each day
# This creates a new DataFrame with a unique daily index, suitable for time series calculations.
merged_data_daily = merged_data.resample('D').last().dropna()

# Calculate daily log returns for the TWD/USD exchange rate
# Use np.log for correct log returns
merged_data_daily['USD_per_TWD_Log_Return'] = np.log(merged_data_daily['USD_per_TWD_Rate'] / merged_data_daily['USD_per_TWD_Rate'].shift(1))

# Define the window for realized volatility (e.g., X days for approximately one month)
volatility_window = 20

# Calculate the rolling standard deviation of the log returns
# This gives daily volatility
merged_data_daily['Realized_Volatility_Daily'] = merged_data_daily['USD_per_TWD_Log_Return'].rolling(window=volatility_window).std()

# Annualize the daily volatility (assuming X trading days in a year)
merged_data_daily['Realized_Volatility_Annualized'] = merged_data_daily['Realized_Volatility_Daily'] * (252**0.5)

print(f"Realized volatility calculated with a {volatility_window}-day window.")
print(merged_data_daily[['USD_per_TWD_Rate', 'USD_per_TWD_Log_Return', 'Realized_Volatility_Annualized']].tail())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

# Plot using the new merged_data_daily DataFrame
ax.plot(merged_data_daily.index, merged_data_daily['Realized_Volatility_Annualized'], label=f'Annualized Realized Volatility ({volatility_window}-Day Rolling)', color='orange')

ax.set_title(f'Realized Volatility of USD/TWD Exchange Rate ({volatility_window}-Day Rolling)')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Realized Volatility Analysis for TSMC Taiwan and TSM ADR

In [ ]:
# Calculate daily log returns for TSMC Taiwan (X)
merged_data_daily['TSMC_TW_Log_Return'] = np.log(merged_data_daily['TSMC_TW_Close'] / merged_data_daily['TSMC_TW_Close'].shift(1))

# Calculate daily log returns for TSM ADR (in TWD)
merged_data_daily['TSM_ADR_TWD_Log_Return'] = np.log(merged_data_daily['TSM_ADR_Close_TWD'] / merged_data_daily['TSM_ADR_Close_TWD'].shift(1))

# Define the window for realized volatility (reusing 'volatility_window' from previous step)

# Calculate the rolling standard deviation of log returns for TSMC Taiwan
merged_data_daily['TSMC_TW_Realized_Volatility_Daily'] = merged_data_daily['TSMC_TW_Log_Return'].rolling(window=volatility_window).std()

# Annualize the daily volatility for TSMC Taiwan
merged_data_daily['TSMC_TW_Realized_Volatility_Annualized'] = merged_data_daily['TSMC_TW_Realized_Volatility_Daily'] * (252**0.5)

# Calculate the rolling standard deviation of log returns for TSM ADR
merged_data_daily['TSM_ADR_TWD_Realized_Volatility_Daily'] = merged_data_daily['TSM_ADR_TWD_Log_Return'].rolling(window=volatility_window).std()

# Annualize the daily volatility for TSM ADR
merged_data_daily['TSM_ADR_TWD_Realized_Volatility_Annualized'] = merged_data_daily['TSM_ADR_TWD_Realized_Volatility_Daily'] * (252**0.5)

print(f"Realized volatility for TSMC Taiwan and TSM ADR calculated with a {volatility_window}-day window.")
print(merged_data_daily[['TSMC_TW_Close', 'TSMC_TW_Realized_Volatility_Annualized', 'TSM_ADR_Close_TWD', 'TSM_ADR_TWD_Realized_Volatility_Annualized']].tail())

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(merged_data_daily.index, merged_data_daily['TSMC_TW_Realized_Volatility_Annualized'], label=f'TSMC (Taiwan) Realized Volatility ({volatility_window}-Day Rolling)', color='blue')
ax.plot(merged_data_daily.index, merged_data_daily['TSM_ADR_TWD_Realized_Volatility_Annualized'], label=f'TSM ADR (TWD) Realized Volatility ({volatility_window}-Day Rolling)', color='green', linestyle='--')

ax.set_title(f'Realized Volatility Comparison: TSMC Taiwan vs. TSM ADR (TWD) ({volatility_window}-Day Rolling)')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Combined Realized Volatility Plot: Stocks and Exchange Rate

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 7))

# Plot TSMC Taiwan and TSM ADR volatilities on the primary y-axis (axX)
ax1.plot(merged_data_daily.index, merged_data_daily['TSMC_TW_Realized_Volatility_Annualized'],
         label=f'TSMC (Taiwan) Realized Volatility ({volatility_window}-Day Rolling)', color='blue')
ax1.plot(merged_data_daily.index, merged_data_daily['TSM_ADR_TWD_Realized_Volatility_Annualized'],
         label=f'TSM ADR (TWD) Realized Volatility ({volatility_window}-Day Rolling)', color='green', linestyle='--')

ax1.set_xlabel('Date')
ax1.set_ylabel('Annualized Stock Volatility', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.grid(True)

# Create a secondary y-axis (axX) for the TWD/USD exchange rate volatility
ax2 = ax1.twinx()
ax2.plot(merged_data_daily.index, merged_data_daily['Realized_Volatility_Annualized'],
         label=f'USD/TWD Exchange Rate Realized Volatility ({volatility_window}-Day Rolling)', color='red', linestyle=':')
ax2.set_ylabel('Annualized FX Volatility', color='red')
ax2.tick_params(axis='y', labelcolor='red')

# Add title and combined legend
plt.title(f'Realized Volatility Comparison: Stocks vs. USD/TWD Exchange Rate ({volatility_window}-Day Rolling)')

# Combine legends from both axes
lines, labels = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc='upper left')

plt.tight_layout()
plt.show()

### Integrating FX Volatility with TSM ADR Volatility

To understand the total volatility of an investment in TSM ADR from a TWD perspective, it's crucial to integrate the volatility of the ADR's price (in USD) with the volatility of the TWD/USD exchange rate. The overall volatility of a foreign asset converted to local currency can be approximated using the following formula for annualised variances of log returns:

$\sigma^X_{ADR_{TWD}} \approx \sigma^X_{ADR_{USD}} + \sigma^X_{FX} + X \times \rho_{ADR_{USD}, FX} \times \sigma_{ADR_{USD}} \times \sigma_{FX}$

Where:
-   $\sigma_{ADR_{TWD}}$: Annualized volatility of TSM ADR in TWD.
-   $\sigma_{ADR_{USD}}$: Annualized volatility of TSM ADR in USD.
-   $\sigma_{FX}$: Annualized volatility of the TWD/USD exchange rate.
-   $\rho_{ADR_{USD}, FX}$: Rolling correlation between the daily log returns of TSM ADR (USD) and the TWD/USD exchange rate.

We will now calculate the necessary components ($\sigma_{ADR_{USD}}$ and $\rho_{ADR_{USD}, FX}$) and then use this formula to derive an 'integrated' TSM ADR TWD volatility to compare with the directly calculated one.

#### Calculate TSM ADR (USD) Volatility

In [ ]:
import numpy as np

# Calculate daily log returns for TSM ADR in USD
merged_data_daily['TSM_ADR_USD_Log_Return'] = np.log(merged_data_daily['TSM_ADR_Close_USD'] / merged_data_daily['TSM_ADR_Close_USD'].shift(1))

# Calculate the rolling standard deviation of log returns for TSM ADR (USD)
merged_data_daily['TSM_ADR_USD_Realized_Volatility_Daily'] = merged_data_daily['TSM_ADR_USD_Log_Return'].rolling(window=volatility_window).std()

# Annualize the daily volatility for TSM ADR (USD)
merged_data_daily['TSM_ADR_USD_Realized_Volatility_Annualized'] = merged_data_daily['TSM_ADR_USD_Realized_Volatility_Daily'] * (252**0.5)

print(f"Realized volatility for TSM ADR (USD) calculated with a {volatility_window}-day window.")
print(merged_data_daily[['TSM_ADR_Close_USD', 'TSM_ADR_USD_Realized_Volatility_Annualized']].tail())

#### Calculate Rolling Covariance and Correlation

In [ ]:
import numpy as np

# Calculate rolling covariance between TSM ADR (USD) log returns and USD/TWD log returns
merged_data_daily['Rolling_Covariance_ADR_USD_FX'] = merged_data_daily['TSM_ADR_USD_Log_Return'].rolling(window=volatility_window).cov(merged_data_daily['USD_per_TWD_Log_Return'])

# Calculate rolling correlation
# Correlation = Covariance / (Std_Dev_ADR_USD * Std_Dev_FX)
merged_data_daily['Rolling_Correlation_ADR_USD_FX'] = merged_data_daily['Rolling_Covariance_ADR_USD_FX'] / \
                                                    (merged_data_daily['TSM_ADR_USD_Realized_Volatility_Daily'] * \
                                                     merged_data_daily['Realized_Volatility_Daily'])

print(f"Rolling covariance and correlation between TSM ADR (USD) and FX returns calculated with a {volatility_window}-day window.")
print(merged_data_daily[['TSM_ADR_USD_Log_Return', 'USD_per_TWD_Log_Return', 'Rolling_Covariance_ADR_USD_FX', 'Rolling_Correlation_ADR_USD_FX']].tail())

#### Calculate Integrated TSM ADR (TWD) Volatility

In [ ]:
import numpy as np

# Extract the annualized volatilities for the calculation
sigma_adr_usd_sq = merged_data_daily['TSM_ADR_USD_Realized_Volatility_Annualized']**2
sigma_fx_sq = merged_data_daily['Realized_Volatility_Annualized']**2

sigma_adr_usd = merged_data_daily['TSM_ADR_USD_Realized_Volatility_Annualized']
sigma_fx = merged_data_daily['Realized_Volatility_Annualized']
rho_adr_usd_fx = merged_data_daily['Rolling_Correlation_ADR_USD_FX']

# Apply the combined volatility formula
merged_data_daily['TSM_ADR_TWD_Volatility_Integrated'] = np.sqrt(sigma_adr_usd_sq + sigma_fx_sq + \
                                                              2 * rho_adr_usd_fx * sigma_adr_usd * sigma_fx)

print("Integrated TSM ADR (TWD) volatility calculated.")
print(merged_data_daily[['TSM_ADR_TWD_Realized_Volatility_Annualized', 'TSM_ADR_TWD_Volatility_Integrated']].tail())

#### Compare Directly Calculated vs. Integrated TSM ADR (TWD) Volatility

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(14, 7))

ax.plot(merged_data_daily.index, merged_data_daily['TSM_ADR_TWD_Realized_Volatility_Annualized'],
         label=f'TSM ADR (TWD) Realized Volatility (Direct Calculation)', color='green')
ax.plot(merged_data_daily.index, merged_data_daily['TSM_ADR_TWD_Volatility_Integrated'],
         label=f'TSM ADR (TWD) Realized Volatility (Integrated FX Component)', color='purple', linestyle='--')

ax.set_title(f'Comparison: TSM ADR (TWD) Volatility - Direct vs. Integrated ({volatility_window}-Day Rolling)')
ax.set_xlabel('Date')
ax.set_ylabel('Annualized Volatility')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

### Interpretation of Integrated Volatility:

The plot above compares the two ways of calculating TSM ADR's volatility in TWD:

X  **Direct Calculation (`TSM_ADR_TWD_Realized_Volatility_Annualized`)**: This was derived directly from the log returns of the TSM ADR price already converted to TWD. It inherently captures both the ADR's price movements in USD and the currency conversion effect.
X  **Integrated Calculation (`TSM_ADR_TWD_Volatility_Integrated`)**: This was calculated by explicitly combining the volatility of the TSM ADR in USD, the volatility of the TWD/USD exchange rate, and their rolling correlation.

The close alignment of these two lines on the plot confirms that the formula for integrating volatilities is a valid way to conceptualize the total risk. It demonstrates that the volatility of the TSM ADR when viewed in TWD is indeed a function of its own USD-denominated volatility, the FX volatility, and the interplay (correlation) between these two factors. This breakdown is particularly useful for risk managers and investors seeking to understand the sources of their portfolio's volatility and how currency movements contribute to the overall risk of holding foreign-denominated assets or their derivatives like ADRs.

In [ ]:
# Save the merged_data_daily DataFrame containing all volatility data to a CSV file
output_filename = 'volatility_data.csv'
merged_data_daily.to_csv(output_filename)

print(f"Volatility data saved to {output_filename}")

**Reasoning**:
The subtask requires selecting specific columns, handling missing values, verifying the DataFrame, and then displaying its head and info. This code block will perform all these actions sequentially.



In [ ]:
columns_for_correlation = [
    'TSMC_TW_Close',
    'TSM_ADR_Close_USD',
    'USD_per_TWD_Rate',
    'TSM_ADR_Close_TWD',
    'Difference_TWD',
    'Realized_Volatility_Annualized',
    'TSMC_TW_Realized_Volatility_Annualized',
    'TSM_ADR_TWD_Realized_Volatility_Annualized'
]

# Calculate Difference_TWD on merged_data_daily as it was lost during resampling
merged_data_daily['Difference_TWD'] = merged_data_daily['TSM_ADR_Close_TWD'] - merged_data_daily['TSMC_TW_Close']

# Create a new DataFrame with selected columns
correlation_data = merged_data_daily[columns_for_correlation].copy()

# Drop any rows with missing values
initial_rows = len(correlation_data)
correlation_data.dropna(inplace=True)
cleaned_rows = len(correlation_data)

print(f"Dropped {initial_rows - cleaned_rows} rows with missing values.")

# Verify that the correlation_data DataFrame is not empty
if correlation_data.empty:
    print("Warning: The correlation_data DataFrame is empty after cleaning. No data available for correlation analysis.")
else:
    print("Correlation data prepared successfully. Displaying first 5 rows and info:")
    print(correlation_data.head())
    print("\nDataFrame Info:")
    correlation_data.info()

### Correlation Matrix Calculation and Visualization

Now that the relevant data is prepared, we will calculate the Pearson correlation matrix for all selected variables. The correlation matrix will show the linear relationships between each pair of variables. We will then visualize this matrix using a heatmap for easy interpretation. A heatmap provides a clear visual representation of correlation strengths and directions (positive or negative).

**Reasoning**:
Following the explanation of the correlation matrix, I will now generate the Python code to calculate the correlation matrix for the `correlation_data` DataFrame and visualize it using a heatmap, fulfilling the next part of the subtask.



In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix
correlation_matrix = correlation_data.corr()

print("Correlation Matrix:")
print(correlation_matrix)

# Plot the correlation matrix using a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of TSMC, TSM ADR, and TWD/USD Exchange Rate Metrics')
plt.show()

print("Correlation matrix calculated and visualized successfully.")

### Interpretation of the Correlation Matrix

The correlation matrix provides valuable insights into the relationships between the prices and volatilities of TSMC Taiwan, TSM US ADR, and the TWD/USD exchange rate. Here are the key findings:

**X Strong Positive Correlation between TSMC Taiwan and TSM ADR Prices:**
-   `TSMC_TW_Close` and `TSM_ADR_Close_USD` show a very strong positive correlation (X).
-   `TSMC_TW_Close` and `TSM_ADR_Close_TWD` also show a very strong positive correlation (X).
-   This indicates that the prices of TSMC's Taiwan-listed shares and its US-listed ADRs move almost perfectly in tandem. This is expected, as the ADR is a derivative instrument representing ownership in the underlying Taiwan shares, and arbitrage mechanisms tend to keep their prices closely aligned.

**X High Correlation of Difference_TWD with Prices:**
-   `Difference_TWD` (TSM ADR in TWD - TSMC TW Close) shows a strong positive correlation with both `TSM_ADR_Close_USD` (X) and `TSM_ADR_Close_TWD` (X). It also has a strong positive correlation with `TSMC_TW_Close` (X).
-   This suggests that as the prices of both the Taiwan stock and the ADR increase, the absolute premium/discount (the dollar difference) between them also tends to increase. This could be due to scaling effects where larger absolute prices lead to larger absolute differences, even if the percentage difference remains relatively stable.

**X TWD/USD Exchange Rate's Relationship with Prices:**
-   `USD_per_TWD_Rate` (which is USD per X TWD, meaning an increase implies TWD appreciation) has a positive but weaker correlation with the stock prices (`TSMC_TW_Close`: X, `TSM_ADR_Close_USD`: X, `TSM_ADR_Close_TWD`: X).
-   This implies that while there's some positive relationship (TWD appreciation might coincide with higher stock prices, or vice-versa), it's not a dominant factor compared to the direct correlation between the stocks themselves. The correlation with `TSM_ADR_Close_USD` is slightly higher, which might reflect the direct currency conversion impact on the USD-denominated ADR.

**X Volatility Relationships:**
-   `Realized_Volatility_Annualized` (USD/TWD FX volatility) shows a moderate positive correlation with `USD_per_TWD_Rate` (X). This suggests that periods of higher exchange rate volatility tend to occur when the TWD is strengthening against the USD.
-   The stock volatilities (`TSMC_TW_Realized_Volatility_Annualized` and `TSM_ADR_TWD_Realized_Volatility_Annualized`) are positively correlated with each other (X), indicating that the volatility of the underlying stock and its ADR move in the same direction, which is expected.
-   Interestingly, there are negative correlations between stock prices and stock volatilities (e.g., `TSMC_TW_Close` and `TSMC_TW_Realized_Volatility_Annualized` is -X). This could hint at a leverage effect, where increasing stock prices are associated with decreasing volatility, and vice-versa, though the correlation is weak.
-   A more pronounced negative correlation exists between the `USD_per_TWD_Rate` and the stock volatilities (`TSMC_TW_Realized_Volatility_Annualized`: -X, `TSM_ADR_TWD_Realized_Volatility_Annualized`: -X). This implies that when the TWD appreciates (higher `USD_per_TWD_Rate`), stock volatilities tend to be lower, and vice versa. This could be a significant finding for risk management and understanding cross-asset dynamics.

**Implications:**
-   **Arbitrage Efficiency:** The extremely high correlation between TSMC Taiwan and TSM ADR prices confirms the efficiency of arbitrage mechanisms in linking these two markets. Any significant divergence would likely be quickly corrected.
-   **Currency Risk Impact:** While the TWD/USD exchange rate does not have an overwhelmingly strong direct correlation with stock prices, its volatility has a notable inverse relationship with stock volatilities. This suggests that currency fluctuations can play a role in the overall risk profile of TSMC investments, particularly for TSM ADR holders.
-   **Hedging Considerations:** For investors holding the TSM ADR, the currency correlation and volatility relationships are crucial. The ADR's price is directly affected by the USD/TWD rate, and the volatility of the FX market may impact the volatility of the ADR itself.

Overall, the analysis confirms strong interconnectedness, particularly between the two stock listings, and reveals interesting inverse relationships between currency movements and stock volatilities.

# Task
Analyze the correlations between the prices and volatilities of TSMC Taiwan, TSM US ADR, and the TWD/USD exchange rate. First, select the relevant price and volatility columns from the `merged_data_daily` DataFrame, ensuring all are numeric and handling any missing values. Then, calculate and visualize the correlation matrix using a heatmap, and finally, interpret the findings, highlighting strong relationships and their implications.

## Prepare Data for Correlation (Revised)

### Subtask:
Select the relevant price and volatility columns from the `merged_data_daily` DataFrame, ensuring all are numeric and handling any missing values. Also, verify that the DataFrame is not empty after cleaning.


## Final Task

### Subtask:
Provide an interpretation of the correlation analysis, highlighting strong positive and negative correlations between prices and volatilities, and discuss what these relationships might imply.


## Summary:

### Q&A
The correlation analysis revealed the following key relationships and their implications:
*   **Strong positive correlation** between `TSMC_TW_Close` and `TSM_ADR_Close_USD` (X), and between `TSMC_TW_Close` and `TSM_ADR_Close_TWD` (X) implies high market efficiency and effective arbitrage between the Taiwan-listed shares and US-listed ADRs.
*   **High positive correlation** of `Difference_TWD` (TSM ADR in TWD - TSMC TW Close) with all price columns (X to X) suggests that as stock prices increase, the absolute premium/discount between the ADR and the Taiwan share also tends to widen.
*   The `USD_per_TWD_Rate` has a **positive but weaker correlation** with stock prices (X to X), indicating it's not a primary driver of stock price movements compared to the direct stock-to-stock relationships.
*   **Stock volatilities** (`TSMC_TW_Realized_Volatility_Annualized` and `TSM_ADR_TWD_Realized_Volatility_Annualized`) are positively correlated with each other (X), as expected.
*   A **moderate positive correlation** (X) exists between `Realized_Volatility_Annualized` (FX volatility) and `USD_per_TWD_Rate`, suggesting that higher exchange rate volatility occurs when the TWD is strengthening.
*   A **pronounced negative correlation** was found between `USD_per_TWD_Rate` and stock volatilities (-X for TSMC TW and -X for TSM ADR), implying that TWD appreciation tends to coincide with lower stock volatilities. This is a significant finding for risk management and cross-asset dynamics.

### Data Analysis Key Findings
*   TSMC Taiwan shares (`TSMC_TW_Close`) and TSM US ADRs (`TSM_ADR_Close_USD`, `TSM_ADR_Close_TWD`) exhibit extremely strong positive correlations, ranging from X to X, indicating highly synchronized price movements.
*   The `Difference_TWD` (premium/discount of TSM ADR in TWD over TSMC TW) shows strong positive correlations with all price metrics (X to X), suggesting that absolute price differences grow with increasing stock prices.
*   The `USD_per_TWD_Rate` has a weak positive correlation with stock prices (X to X), implying a limited direct impact on stock price movements.
*   Stock volatilities (`TSMC_TW_Realized_Volatility_Annualized` and `TSM_ADR_TWD_Realized_Volatility_Annualized`) are positively correlated with each other at X.
*   A moderate positive correlation of X exists between the `Realized_Volatility_Annualized` (FX volatility) and `USD_per_TWD_Rate`.
*   A significant negative correlation was observed between `USD_per_TWD_Rate` and stock volatilities (ranging from -X to -X), implying that an appreciating TWD is associated with lower stock volatilities.

### Insights or Next Steps
*   The exceptionally strong correlation between TSMC's Taiwan listing and its US ADR confirms efficient arbitrage, minimizing price discrepancies across markets. This reinforces the idea that investors generally perceive them as interchangeable assets, with price deviations quickly corrected.
*   The inverse relationship between TWD appreciation and stock volatilities suggests that currency movements are an important factor in assessing and managing the risk profile of TSMC investments, particularly for TSM ADR holders who have direct currency exposure. Further investigation into the causality of this relationship could provide valuable insights for hedging strategies.
